# Lecture 5 — Distribution Charts (Histogram, Box Plot & Violin Plot

> **Dataset:** Airbnb London Listings (https://insideairbnb.com/get-the-data/)

> **Source of figures:** Knaflic, C. N. (2015). *Storytelling with data*. Wiley.



---
## Opening: Model Answer Review 

---
## Design Principles - Part I: Text is Your Friend


*Ticket chart with titles and axis labels added — text makes it accessible [ FIGURE 5.9 — p.144 ]*

>  <img src="attachment:ff37f6ed-f362-433e-af91-51884a95a232.png" width="800"/> 



**[ FIGURE 5.10 — p.144 ]**

*Same chart with action title and annotation — the call to action is now clear [ FIGURE 5.10 — p.144 ]*

> <img src="attachment:d66f452a-b15a-4f21-a300-e09cdca7ec6c.png" width="800"/> 

----
## Design Principles - Part II: Why Distributions?


### The mean hides everything

> ❓ **The average Airbnb price in London is £110 per night. Is that useful information? If I told you some listings cost £30 and others cost £800, does £110 feel like the right number to plan around?**

> 💡 **The mean collapses the entire shape of the data into one number — and for skewed data like prices, it is almost always misleading
The mean collapses the entire shape of the data into one number — and for skewed data like prices, it is almost always misleading**

> 💡 **median, range, IQR, what % are below £100, etc. offer a more accurate representation of the data**

**Three charts for understanding distributions:**

| Chart | Best for | When to use |
|---|---|---|
| Histogram | Shape of ONE distribution | Where are the values concentrated? |
| Box plot | Comparing MANY groups compactly | Quick comparison of medians + spread |
| Violin plot | Comparing groups WITH shape | When shape matters (bimodal, skewed) |


> 💡**Distribution charts are often the hardest to title well because the shape IS the story. You need words to interpret the shape for your audience. Don't make them figure out what the skew means — tell them**

**Text rules for distribution charts:**
- Always name the distribution shape in the title ('right-skewed', 'concentrated around', 'bimodal')
- Add a vertical line for mean and/or median — annotate which is which
- Explain outliers: are they errors or interesting? Annotate them


---
## Let's Code Some Examples 💻 
|


In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Airbnb London Listings

df = pd.read_csv('../data/airbnb_london.csv')
print(f"Loaded: {len(df)} listings")
print(df.describe().round(1))


Loaded: 2500 listings
        price  minimum_nights  number_of_reviews  availability_365  \
count  2500.0          2500.0             2500.0            2500.0   
mean    148.6            14.8              147.9             183.7   
std     110.9             8.4               86.3             105.5   
min      20.5             1.0                0.0               0.0   
25%      71.7             8.0               74.0              92.0   
50%     117.5            15.0              145.0             182.0   
75%     188.9            22.0              222.2             277.0   
max    1032.4            29.0              299.0             364.0   

       reviews_per_month  
count             2500.0  
mean                 2.0  
std                  2.0  
min                  0.0  
25%                  0.6  
50%                  1.4  
75%                  2.8  
max                 15.2  


### Example 1 — Histogram: price distribution

In [2]:
# explorative plot to look at price distribution

fig = px.histogram(df, x='price', title='Price Distribution', height=500, nbins=50)
fig.show()


In [3]:
# Let's improve it: handling the outliers problem
# cap at 95th percentile so the bulk of data is visible (Always disclose this in annotation!!!) + add median line for reference

p95 = df['price'].quantile(0.95)
df_cap = df.loc[df['price'] <= p95].copy()

fig = px.histogram(
    data_frame=df_cap, 
    x='price',
    nbins=50,                              # 50 bins: enough granularity to see shape
    color_discrete_sequence=['#2E75B6'],   
    labels={'price': 'Nightly Price (£)', 'count': 'Number of Listings'},
    height=500
)

# Add median line
median_price = df_cap['price'].median()

fig.add_vline(x=median_price, line_dash='dash', line_color='#E63946', line_width=2,
              annotation=dict(
                  text=f'<b>Median: £{median_price:.0f}',
                  font=dict(color='#E63946', size=12),
                  xanchor='left',
                  yanchor='top',
                  xshift=8,   # pixels to the right of the line
              ))

fig.update_layout(
    title=f'London Airbnb prices are heavily right-skewed — most listings are under £150, but outliers reach £{p95:.0f}+',
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=13),
    yaxis=dict(gridcolor='#EEEEEE', title='Number of Listings'),
    xaxis=dict(showgrid=False),
    margin=dict(l=60, r=40, t=65, b=40)
)
fig.show()

### Example 2 — Box plot: comparing room types

In [4]:
# exploratory viz to check price distribution per listing type

fig = px.box(df, x='room_type', y='price', title='Price by Room Type', height=500)
fig.show()

In [5]:
# ── IMPROVED: cap, colour by room type with meaning, notched boxes ────────

# Let's improve it: use 95% capped data, used notched boxes. Color is an option here and depends on the context

fig = px.box(
    data_frame=df_cap, 
    y='room_type', x='price',
    color='room_type',
    # Meaningful colour: entire home (most expensive) gets the standout colour
    color_discrete_map={
        'Entire home/apt': '#2E75B6',
        'Private room': '#70AD47',
        'Shared room': '#AAAAAA'
    },
    notched=True,                  # notch shows confidence interval around median
    points='outliers',             # show outliers as individual points
    labels={'price': 'Nightly Price (£)', 'room_type': ''},
    category_orders={'room_type': ['Entire home/apt', 'Private room', 'Shared room']}, height=500
)

fig.update_layout(
    title='Entire homes cost 2x private rooms — shared rooms are cheapest but',
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=13),
    xaxis=dict(gridcolor='#EEEEEE', title='Nightly Price (£)'),
    yaxis=dict(showgrid=False),
    showlegend=False,
    margin=dict(l=60, r=40, t=55, b=40)
)
fig.show()


### Example 3 — Violin plot: comparing distributions with shape

In [6]:
# Explorative viz for price dist. by neighbourhood

fig = px.violin(data_frame=df, 
                x='price', y='neighbourhood', 
                title='Price by Neighbourhood',
               height=900, width=1000)
fig.show()


In [7]:
# ── IMPROVED: horizontal, sorted by median, highlight expensive areas ─────
# Compute median per neighbourhood for sorting

med_order = df_cap.groupby('neighbourhood')['price'].median().sort_values(ascending=True).index.tolist()

fig = px.violin(
    df_cap,
    y='neighbourhood',           # horizontal makes long labels readable
    x='price',
    color='neighbourhood',
    orientation='h',
    box=True,                    # embed box plot inside violin
    points=False,                # no individual points — violin shows shape already
    category_orders={'neighbourhood': med_order},
    color_discrete_sequence=['#DDDDDD']*len(med_order),  # all grey first
    height=900, width=1000    
)

# Highlight the two most expensive neighbourhoods
expensive = ['Kensington and Chelsea', 'Westminster']
for trace in fig.data:
    if trace.name in expensive:
        trace.line.color = 'rgba(230,57,70,1)'
        trace.fillcolor = 'rgba(230,57,70,0.25)' # lighter shade of the line color using a = opacity
        
cheap = ['Lambeth', 'Tower Hamlets']
for trace in fig.data:
    if trace.name in cheap:
        trace.line.color = 'rgb(46, 117, 182)'
        trace.fillcolor = 'rgba(46, 117, 182, 0.25)'

fig.update_layout(
    title='Kensington and Westminster command premium prices — Hackney and Lambeth are most affordable',
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    xaxis=dict(gridcolor='#EEEEEE', title='Nightly Price (£)'),
    yaxis=dict(showgrid=False, title=''),
    showlegend=False,
    margin=dict(l=180, r=40, t=55, b=40)
)
fig.show()


---
## Class Exercise 💪 💻

**Checklist:**

- Cap extreme outliers and disclose it in annotation
- Add median/mean reference line on histograms
- Insight title that names the distribution shape
- Colour used meaningfully (one category = one colour)
